# Import Data After Cleaning

In [47]:
import pandas as pd
df = pd.read_csv("RetailCleaned.csv", header = None)
df.columns = ["Invoice", "StockCode", "Description","Quantity","InvoiceDate","Price","Customer_ID","Country", "SourceSheet"]
df["InvoiceDate"]= pd.to_datetime(df["InvoiceDate"])
print(df) 

C:\Users\mdserajanwar\AppData\Local\Temp\ipykernel_1636\156276391.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("RetailCleaned.csv", header = None)


         Invoice StockCode                         Description  Quantity  \
0         489434     21232      STRAWBERRY CERAMIC TRINKET BOX        24   
1         489434     21523  FANCY FONT HOME SWEET HOME DOORMAT        10   
2         489434     21871                 SAVE THE PLANET MUG        24   
3         489434     22041        RECORD FRAME 7" SINGLE SIZE         48   
4         489434     22064          PINK DOUGHNUT TRINKET POT         24   
...          ...       ...                                 ...       ...   
1033031  C581490     23144     ZINC T-LIGHT HOLDER STARS SMALL       -11   
1033032  C581499         M                              Manual        -1   
1033033  C581568     21258          VICTORIAN SEWING BOX LARGE        -5   
1033034  C581569     20979       36 PENCILS TUBE RED RETROSPOT        -5   
1033035  C581569     84978    HANGING HEART JAR T-LIGHT HOLDER        -1   

                InvoiceDate   Price  Customer_ID         Country SourceSheet  
0       

# Data Exploring 

In [48]:
# df.shape
# df.head(5)
# df.info()
# df.describe()
# df.isnull().sum()
# df.duplicated().sum()
df.tail(5) 

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet
1033031,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom,2010-11
1033032,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom,2010-11
1033033,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom,2010-11
1033034,C581569,20979,36 PENCILS TUBE RED RETROSPOT,-5,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,2010-11
1033035,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,2010-11


# Filter and Calculation Recency, Frequency, Monetary

In [49]:
rfm_data = df[
    (df['Customer_ID'].notnull()) &
    (df['Quantity'] > 0)
].copy()

snapshot_date = pd.to_datetime("2011-12-10")

rfm = rfm_data.groupby('Customer_ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index() 

# Score Customers and Business Segments

In [53]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels = [5, 4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method = "first"), 5, labels = [1,2,3,4,5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 5, labels = [1,2,3,4,5]).astype(int)

def segment_customer(row):

    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "Champions"

    elif row["R_Score"] >= 3 and row["F_Score"] >= 3:
        return "Loyal"

    elif row["R_Score"] <= 2 and row["F_Score"] >= 3:
        return "At Risk"

    elif row["R_Score"] >= 4 and row["F_Score"] <= 2:
        return "New"

    elif row["R_Score"] == 3 and row["F_Score"] <= 2:
        return "Needs Attention"

    else:
        return "Lost"

rfm["Segment"] = rfm.apply(segment_customer, axis = 1 )
rfm["IsChurned"] = rfm["Recency"] > 90

# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT

In [60]:
df = df[(df['Customer_ID'].notna()) &(df['Quantity'] > 0)].copy()

df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
    rfm[['Customer_ID', 'Recency', 'Frequency', 'Monetary','R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='Customer_ID',
    how='left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index=False)